# Évaluation des modèles — Risque 5 km (CG + IC) & gain de temps

Reprend le protocole officiel du data battle (`Evaluation_databattle_meteorage.ipynb`)
mais **étend la zone dangereuse à 5 km** et **décompose le risque par type d'éclair**
(CG = cloud-to-ground, qui touche le sol ; IC = intra-cloud).

**Données** : CSV d'éval officiel tenu à l'écart (saisons 2023+).
**Modèles** : ré-inférence depuis les checkpoints sauvegardés dans `models/`
(à produire au préalable via les notebooks d'entraînement). Les modèles absents
sont ignorés. **À exécuter sur un env GPU** (l'inférence séquentielle MC Dropout
est coûteuse en CPU).

### Définitions
- **Gain** : pour chaque alerte couverte, \((t_{last} + 30\,\text{min}) - t_{pred}\), sommé.
  \(t_{last}\) = dernier éclair de l'alerte (CG ou IC), \(t_{pred}\) = fin d'alerte prédite.
- **Risque** : \(R = M / N\) dans la zone \(dist < 5\) km, où \(N\) = nb total d'éclairs
  et \(M\) = nb d'éclairs **manqués** (survenus après \(t_{pred}\), donc entre \(t_{pred}\)
  et \(t_{last}\)). Calculé en global (CG+IC) **et** décomposé en \(R_{CG}\), \(R_{IC}\).
- **Sélection** : à un seuil de confiance \(\theta\), on retient par alerte la fin prédite
  la plus précoce parmi les prédictions de confiance \(\geq \theta\). Le meilleur \(\theta\)
  est celui de gain maximal respectant \(R < R_{accept}\) (2 %).


## 0. Configuration

Adapter `EVAL_CSV` et `MODELS_DIR` si les chemins diffèrent sur l'env GPU.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Le notebook peut être lancé depuis notebooks/ ou depuis la racine du repo.
CWD = Path.cwd()
REPO_DIR = CWD if (CWD / "src").exists() else CWD.parent
sys.path.insert(0, str(REPO_DIR / "src"))

import risk_gain_5km as rg
import eval_inference as ei

# ── Chemins (À ADAPTER sur l'env GPU si besoin) ───────────────────────────────
EVAL_CANDIDATES = [
    Path.home() / "Bureau/Hackathon/hackathon2026/data/segment_alerts_all_airports_eval.csv",
    REPO_DIR / "data/segment_alerts_all_airports_eval.csv",
]
EVAL_CSV = next((p for p in EVAL_CANDIDATES if p.exists()), EVAL_CANDIDATES[0])
MODELS_DIR = REPO_DIR / "models"

# ── Paramètres d'évaluation ───────────────────────────────────────────────────
MIN_DIST_KM = 5.0       # zone dangereuse (CG + IC)
ACCEPTABLE_RISK = 0.02  # risque acceptable (cf. protocole jury)

# ── Paramètres d'inférence séquentielle (ajustables sur GPU) ──────────────────
ei.N_MC = 50                                       # passages MC Dropout (incertitude)
ei.FRACTIONS = [0.2, 0.35, 0.5, 0.65, 0.8, 0.9]    # points de prédiction par session

print("Repo       :", REPO_DIR)
print("Eval CSV   :", EVAL_CSV, "| existe:", EVAL_CSV.exists())
present = sorted(p.name for p in MODELS_DIR.glob("*")) if MODELS_DIR.exists() else []
print("Models dir :", MODELS_DIR, "|", present or "(aucun modèle — entraîner d'abord)")
print("Device     :", ei.DEVICE)

## 1. Chargement & normalisation de l'éval (2023+)

Le CSV jury utilise `alert_id` (renommé `airport_alert_id`) et n'a pas de
`lightning_id` (ajouté). On filtre aux éclairs en alerte.

In [ ]:
alerts = ei.load_eval_alerts(EVAL_CSV)

near = alerts["dist"] < MIN_DIST_KM
print(f"{alerts['airport_alert_id'].nunique()} alertes | "
      f"{len(alerts):,} éclairs en zone d'alerte | "
      f"{alerts['airport'].nunique()} aéroports")
print(f"Éclairs < {MIN_DIST_KM:.0f} km : {int(near.sum()):,} "
      f"(CG = {int((near & ~alerts['icloud']).sum()):,}, "
      f"IC = {int((near & alerts['icloud']).sum()):,})")
alerts.head(3)

## 2. Validation de la machinerie — prédictions factices

Smoke test **sans aucun modèle** : on génère 10 prédictions par alerte autour de
la vraie fin (confiance croissante), façon Part 1 du notebook jury. Cela valide le
calcul gain/risque 5 km CG+IC sur les vraies données d'éval et montre le tradeoff
attendu (θ↑ → gain↓, risque↓).

In [ ]:
last_lightning = pd.to_datetime(
    alerts.groupby(["airport", "airport_alert_id"]).date.max(), utc=True
)
fake = pd.DataFrame(
    [
        (ap, aid,
         tms + pd.to_timedelta(i - 20, unit="m"),  # date d'émission
         tms + pd.to_timedelta(i, unit="m"),       # fin prédite
         (i + 20) / 40)                            # confiance ∈ [0, 1[
        for (ap, aid), tms in last_lightning.items()
        for i in range(-20, 20, 4)
    ],
    columns=rg.PREDICTION_COLS,
)

sweep_fake, totals_fake = rg.evaluate_theta_sweep(fake, alerts, min_dist_km=MIN_DIST_KM)
print("Dénominateurs (éclairs < 5 km) :", totals_fake)
sweep_fake.round(4)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(sweep_fake["risk_total"] * 100, sweep_fake["gain_h"], "*-", label="risque total (CG+IC)")
ax.plot(sweep_fake["risk_cg"] * 100, sweep_fake["gain_h"], "o-", ms=4, label="risque CG seul")
for _, r in sweep_fake.iterrows():
    ax.annotate(f"{r['theta']:.2f}", (r["risk_total"] * 100, r["gain_h"]), fontsize=7)
ax.axvline(ACCEPTABLE_RISK * 100, color="red", ls="--", lw=1,
           label=f"risque acceptable {ACCEPTABLE_RISK*100:.0f}%")
ax.set_xlabel("Risque = éclairs manqués < 5 km (%)")
ax.set_ylabel("Gain de temps (heures)")
ax.set_title("Tradeoff gain / risque — prédictions factices (validation)")
ax.legend()
plt.show()

## 3. Ré-inférence des modèles sauvegardés

⚠️ **Env GPU recommandé.** Charge chaque checkpoint présent dans `models/`, refait
l'inférence sur l'éval et produit les prédictions au format standard. Les modèles
absents sont ignorés. La baseline 30 min est toujours incluse (ancre gain 0).

In [ ]:
predictions = ei.run_all_inference(alerts, models_dir=MODELS_DIR)
print("\nModèles évalués :", list(predictions))

## 4. Évaluation gain / risque par modèle

In [ ]:
sweeps, totals_by_model = {}, {}
for name, preds in predictions.items():
    sweeps[name], totals_by_model[name] = rg.evaluate_theta_sweep(
        preds, alerts, min_dist_km=MIN_DIST_KM
    )
print("Sweeps calculés pour", len(sweeps), "modèles :", list(sweeps))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for name, sw in sweeps.items():
    ax.plot(sw["risk_total"] * 100, sw["gain_h"], "o-", ms=3, label=name)
ax.axvline(ACCEPTABLE_RISK * 100, color="red", ls="--", lw=1,
           label=f"risque acceptable {ACCEPTABLE_RISK*100:.0f}%")
ax.set_xlabel("Risque total = éclairs CG+IC manqués < 5 km (%)")
ax.set_ylabel("Gain de temps (heures)")
ax.set_title(f"Tradeoff gain / risque par modèle (zone {MIN_DIST_KM:.0f} km, CG+IC)")
ax.legend(fontsize=8)
plt.show()

## 5. Meilleur θ et tableau récapitulatif

Pour chaque modèle, le θ de gain maximal respectant **risque total < 2 %**
(métrique jury), avec la décomposition CG / IC à ce θ. `Alertes couvertes` indique
sur combien d'alertes le modèle a pu prédire (les modèles séquentiels ignorent les
sessions à moins de 2 CG).

In [ ]:
rows = []
for name, sw in sweeps.items():
    best = rg.select_best_theta(sw, acceptable_risk=ACCEPTABLE_RISK, risk_col="risk_total")
    if best is None:
        rows.append({"Modèle": name, "θ*": None, "Gain (h)": None,
                     "Risque tot %": None, "Risque CG %": None, "Risque IC %": None,
                     "Manqués CG": None, "Manqués IC": None, "Alertes couvertes": None})
        continue
    rows.append({
        "Modèle": name,
        "θ*": round(best["theta"], 2),
        "Gain (h)": round(best["gain_h"], 1),
        "Risque tot %": round(best["risk_total"] * 100, 2),
        "Risque CG %": round(best["risk_cg"] * 100, 2),
        "Risque IC %": round(best["risk_ic"] * 100, 2),
        "Manqués CG": int(best["missed_cg"]),
        "Manqués IC": int(best["missed_ic"]),
        "Alertes couvertes": int(best["n_alerts_covered"]),
    })
summary = pd.DataFrame(rows).sort_values("Gain (h)", ascending=False, na_position="last")
print(f"Meilleur θ sous contrainte risque total < {ACCEPTABLE_RISK*100:.0f}% "
      f"(zone {MIN_DIST_KM:.0f} km, CG+IC)")
summary

### Variante : contrainte sur le risque CG seul

Les CG touchent le sol et sont les plus dangereux pour l'aéroport. Si l'on impose
**risque CG < 2 %** (plus strict), le θ retenu est en général plus élevé et le gain
plus faible.

In [ ]:
rows_cg = []
for name, sw in sweeps.items():
    best = rg.select_best_theta(sw, acceptable_risk=ACCEPTABLE_RISK, risk_col="risk_cg")
    rows_cg.append({
        "Modèle": name,
        "θ* (CG)": None if best is None else round(best["theta"], 2),
        "Gain (h)": None if best is None else round(best["gain_h"], 1),
        "Risque CG %": None if best is None else round(best["risk_cg"] * 100, 2),
    })
pd.DataFrame(rows_cg).sort_values("Gain (h)", ascending=False, na_position="last")

## Notes & limites

- **Features tabulaires** (XGBoost, BNN) reconstruites sur l'éval avec le `features.py`
  courant : léger écart possible si le parquet d'entraînement a été produit par une
  autre version du feature engineering.
- **Modèles par aéroport** non câblés : seuls les modèles globaux sont chargés.
- **Modèles séquentiels** : une prédiction est émise à chaque fraction d'avancement
  de session (`ei.FRACTIONS`), pas à chaque éclair. Augmenter le nombre de points
  donne plus de chances de trouver une prédiction précoce acceptée.
- **Dénominateur du risque** restreint aux alertes effectivement couvertes par chaque
  modèle (comparaison sur son propre périmètre) → comparer aussi `Alertes couvertes`.
- **t_last vs dernier CG** : les modèles prédisent la fin = dernier CG, alors que le
  risque/gain se mesure jusqu'au dernier éclair (CG ou IC). Un écart est attendu si
  des IC suivent le dernier CG.
- **N_MC / FRACTIONS** ajustables dans la cellule de config (augmenter `N_MC` sur GPU
  pour une incertitude plus stable).
